# CourtVision on Colab

1. **Runtime → Change runtime type → T4 GPU.**
2. In the **Secrets** panel (key icon), add `ROBOFLOW_API_KEY` and `COURTVISION_PASSWORD`, and turn on notebook access for both.
3. Put the roster CSV on Drive at `MyDrive/CS496/courtvision/rosters.csv`. Videos are found anywhere under `MyDrive/CS496/courtvision/` (shortcuts work).

Run the cells from the top. If the install cell says to restart the session, do it and run the cells again.

## 1. Drive and GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/CS496/courtvision'

## 2. Install CourtVision

In [ ]:
import os
from google.colab import userdata

BRANCH = 'dev-jason'  # branch with the latest work
REPO = 'https://github.com/Mythventor/CourtVision.git'

if not os.path.exists('/content/CourtVision'):
    !git clone -q -b {BRANCH} {REPO} /content/CourtVision
else:
    !git -C /content/CourtVision fetch -q && git -C /content/CourtVision checkout -q {BRANCH} && git -C /content/CourtVision pull -q
%cd /content/CourtVision
!python --version
# uv resolves the large inference/torch dependency set more reliably (and faster) than pip
!pip install -q uv
!uv pip install --system -q -e ".[cv,app]"

for name in ('ROBOFLOW_API_KEY', 'COURTVISION_PASSWORD'):
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        print(f'Secret {name} not set')

## 3. Config on Drive

Created once at `MyDrive/CS496/courtvision/config.yaml`. It only needs the settings that differ from `configs/default.yaml`; edit it on Drive (e.g. to add `court.template`).

In [ ]:
CONFIG = f'{DRIVE}/config.yaml'
if not os.path.exists(CONFIG):
    with open(CONFIG, 'w') as f:
        f.write(f'''video_root: {DRIVE}
output_dir: {DRIVE}/outputs
roster_csv: {DRIVE}/rosters.csv
''')
print(open(CONFIG).read())

from courtvision.config import load_config
config = load_config(CONFIG)

## 4. Find the videos

In [ ]:
from courtvision.library import list_videos
videos = list_videos(config['video_root'], exclude=config['output_dir'])
for game, files in videos.items():
    print(game)
    for p in files:
        print('   ', p.name)

VIDEO = str(next(iter(videos.values()))[0]) if videos else None  # change to the quarter you want
print('\nVIDEO =', VIDEO)
!ffprobe -v error -show_entries format=duration:format_tags=creation_time:stream=codec_name,width,height,r_frame_rate -of default=nw=1 "{VIDEO}"

## 5. Check the detection model on one frame

Compare the class names it prints with `detection.classes` in the config.

In [ ]:
import matplotlib.pyplot as plt
from courtvision.preview import preview_detections

T = 120  # seconds into the video
img, classes = preview_detections(VIDEO, T, config)
print('Classes in this frame:', classes)
plt.figure(figsize=(16, 9)); plt.imshow(img); plt.axis('off'); plt.show()

## 6. Court keypoints (one-time setup for the 2D panel)

This numbers the court keypoints the model finds (yellow = confident, red = not). Send the image and the printed list to Claude to fill in `court.template`, or fill it in yourself: one `[x_ft, y_ft]` per keypoint index, `null` for ones to skip.

In [ ]:
from courtvision.preview import preview_keypoints

img, points = preview_keypoints(VIDEO, T, config)
for i, x, y, c in points:
    print(f'{i:2d}: x={x:7.1f} y={y:7.1f} conf={c:.2f}')
plt.figure(figsize=(16, 9)); plt.imshow(img); plt.axis('off'); plt.show()

## 7. Start the app

Open the `https://….gradio.live` link it prints and log in as `courtvision` with your password. Processed videos are saved to `MyDrive/CS496/courtvision/outputs`, so a quarter only has to be processed once. Try a 2-minute range first to check speed.

In [ ]:
!python -m courtvision.app --share --config "{CONFIG}"